# Module 6 — Best practices for a reproducible project

**Who this is for:** anyone starting the capstone, and anyone whose numbers change between runs or machines.

> **Self-check — skip this module if you can already:**
>
> - [ ] set up a virtual environment from a lock file and check the installed versions
> - [ ] keep every setting of a run in one config and fix the seed
> - [ ] add a sanity check that fails loudly when something is wrong
> - [ ] explain what belongs in git and what does not
>
> If you can tick every box, go to the next module. If not, work through the parts you can't tick. Before
> jumping to a section, run the module's **setup cell** (the first code cell): later sections use variables
> defined earlier, so run the cells above a section if you get a `NameError`.

**You will learn**
- environments and version pinning
- one config, one seed
- functions over copy-paste
- sanity checks with `assert`
- units, `dtype` and precision
- git habits for a team
- a debugging routine

**Where the course uses this:** the capstone rubric's *Reproducibility & engineering* criterion and `RESULTS.md`.

*Run each cell with **Shift+Enter**, top to bottom. Nothing here needs data files; once the libraries are
installed (see the [overview](README.md#setup-first)), the notebook runs offline.*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/python_tutorial/06_best_practices.ipynb)
&nbsp; Part of the **Python for biosignal processing** tutorial — see the [overview](README.md) to choose which modules you need.

## 1. Virtual environments: same libraries, same numbers

A **virtual environment** is a private folder with its own Python and its own libraries, used by one project.
It solves three everyday problems:

- **"It works on my laptop but not yours."** Everyone installs the same pinned versions from one file.
- **"Installing X broke Y."** Each project has its own libraries, so they can't break each other.
- **"Something is broken and I don't know why."** You can delete the environment and rebuild it in a minute.

In Colab you don't need one: each session is a fresh machine. On your own computer, use one per project.

### Create, activate, install

```bash
python -m venv .venv                       # create (once per project)
source .venv/bin/activate                  # activate: macOS / Linux
.venv\Scripts\Activate.ps1                # activate: Windows PowerShell (cmd: .venv\Scripts\activate.bat)
pip install -r requirements-lock.txt -r requirements-real.txt
deactivate                                 # leave it
```

When it is active, your prompt shows `(.venv)` and `python` / `pip` point inside the folder. conda
(`conda create -n cm2013 python=3.11`, then `conda activate cm2013`) and `uv` (`uv venv`, then
`uv pip install -r ...`) do the same job; pick one and use it consistently.

In **VS Code**, choose the environment with *Python: Select Interpreter*, and choose it again as the
**kernel** of each notebook (top right). A notebook running on the wrong kernel is the most common reason
for "but I installed it!".

### Check what you have

The cell below shows the Python and library versions **of the kernel running this notebook**:

In [ ]:
import sys, numpy, scipy, sklearn, matplotlib
print("python  ", sys.version.split()[0], "  from", sys.executable)
for m in (numpy, scipy, sklearn, matplotlib):
    print(f"{m.__name__:12s}{m.__version__}")

If `sys.executable` does not point into your `.venv` (or conda environment), the notebook is using a
different Python than the one you installed into.

From a terminal (with the environment active) the useful commands are:

| Command | What it does |
|---|---|
| `pip list` | every installed library and its version |
| `pip show numpy` | one library: version, location, what needs it |
| `pip check` | reports libraries whose requirements conflict |
| `pip install "pandas<3"` | install with a version limit |
| `pip install numpy==2.2.6` | install one exact version |
| `pip freeze > requirements-mine.txt` | write down exactly what you have now |

### Fixing a broken or wrong library version

1. **Read the error.** `ModuleNotFoundError` means it is not installed *in this environment*. An error deep
   inside a library right after an install usually means two libraries' versions don't fit together.
2. **Pin the version that works.** Example from this course: the pinned `wfdb` 4.3.0 fails to import with
   pandas 3, so the course pins `pandas<3`. (Newer `wfdb` releases fix this. Constraints like this belong to
   specific versions; re-test before changing any pin.) Install with `pip install "pandas<3"` and add the
   same line to your requirements file so your teammates get it too.
3. **Check with `pip check`** and by importing the library again.
4. **If it is still a mess, rebuild:** deactivate, delete the `.venv` folder (or `conda env remove -n cm2013`),
   create it again, and install from the lock file. This takes a minute and fixes most problems.
5. **Never `pip install` into your system Python** or with `sudo`. If `pip` complains about an
   "externally managed environment", that is the system telling you to use a virtual environment.

### Pinning for your team

Commit **one** requirements file with exact versions (`numpy==2.2.6`, not `numpy`). Start from the course's
`requirements-lock.txt`, add anything extra your team needs, and pin it. Unpinned requirements are how two
teammates end up with different numbers from the same code.

The course's lock file pins the libraries the notebooks use directly. The libraries *those* depend on can
still vary slightly; `pip freeze > requirements-freeze.txt` records your complete environment if you need
every version written down.

## 2. One config, one seed

Keep every choice that affects the result in one dictionary, print it, and save it next to the results.
Then "what did we run?" always has an answer.

In [ ]:
import numpy as np, json

cfg = {
    "seed": 2013,
    "fs": 100,
    "epoch_s": 30,
    "bandpass_hz": [0.5, 35.0],
    "welch_nperseg_s": 4,
    "classifier": "random_forest",
    "n_estimators": 200,
}
rng = np.random.default_rng(cfg["seed"])
print(json.dumps(cfg, indent=2))

a = np.random.default_rng(cfg["seed"]).standard_normal(3)
b = np.random.default_rng(cfg["seed"]).standard_normal(3)
print("same seed, same numbers:", np.array_equal(a, b))

Pass `cfg["seed"]` to **everything** random: `np.random.default_rng`, `random_state=` in scikit-learn models,
splitters and resamplers. One unseeded step is enough to make results drift. A seed plus pinned versions makes
results repeatable on one machine; on different hardware they can still differ in the last digits (section 5).

## 3. Functions over copy-paste

If you have pasted the same block twice, make it a function. Fixing a bug then fixes it everywhere, and the
function can be tested on its own.

In [ ]:
from scipy import signal

def bandpass(x, fs, lo, hi, order=4):
    """Zero-phase Butterworth band-pass. x: (..., n_samples). Returns the same shape."""
    sos = signal.butter(order, [lo, hi], btype="bandpass", fs=fs, output="sos")
    return signal.sosfiltfilt(sos, x, axis=-1)

fs = cfg["fs"]
t = np.arange(10 * fs) / fs
x = np.sin(2 * np.pi * 10 * t) + np.sin(2 * np.pi * 45 * t)
y = bandpass(x, fs, *cfg["bandpass_hz"])

# sanity check: the 45 Hz part (outside 0.5-35 Hz) should be strongly reduced
f, P_in = signal.welch(x, fs=fs, nperseg=4 * fs)
_, P_out = signal.welch(y, fs=fs, nperseg=4 * fs)
k45 = np.argmin(abs(f - 45))
assert P_out[k45] < 0.01 * P_in[k45], "band-pass did not remove 45 Hz"
print("sanity check passed")

## 4. Sanity checks that fail loudly

The labs end with a *live sanity check*: a few `assert`s on numbers the notebook just computed. Do the same in
your project. Useful checks for biosignal work:

- shapes line up: `assert X.shape[0] == y.shape[0] == groups.shape[0]`
- no subject is in both train and test: `assert set(groups[tr]).isdisjoint(groups[te])`
- no missing values crept in: `assert np.isfinite(X).all()`
- the sampling rate is what you think it is, and the duration matches
- a model beats a trivial baseline before you celebrate it

In [ ]:
groups = np.repeat(np.arange(6), 10)
from sklearn.model_selection import GroupKFold
for tr, te in GroupKFold(3).split(np.zeros(60), groups=groups):
    assert set(groups[tr]).isdisjoint(groups[te]), "subject leakage!"
print("no subject appears in both train and test")

## 5. Units, `dtype` and precision

- **Know your units.** MNE stores EEG in **volts**, so 50 µV is `50e-6`. Plot labels should say µV, mV or V.
- **Integers from an ADC** (`int16`) must be scaled to physical units before filtering. Doing maths on raw
  integers can overflow silently.
- **`float64`** is NumPy's default and the right choice for signal processing. `float32` halves memory but can
  shift results in the last digits.
- **Tiny differences are normal.** The same code can differ in the last digits on different CPUs. Compare
  results with `np.isclose`, not `==`.

In [ ]:
a = np.int16(30000)
print("int16 overflow:", a + np.int16(10000))           # wraps around: not 40000!
print(0.1 + 0.2 == 0.3, np.isclose(0.1 + 0.2, 0.3))

## 6. Git habits for a team

- **Commit small and often**, with a message that says what and why ("iteration 2: add notch before Welch").
- **Do not commit** data (`data_cache/`), virtual environments (`.venv/`), or large outputs. List them in
  `.gitignore`.
- **Commit the lock file and a README** with the exact commands to reproduce your numbers.
- **Everyone commits their own work.** It is the clearest record of who did what.
- **Notebooks and git:** clear outputs before committing, or keep the core code in `.py` files and import it,
  so diffs stay readable.

## 7. A debugging routine

1. **Restart and run all.** Many "bugs" are cells run out of order.
2. **Read the error from the bottom up** and find the line in *your* code.
3. **Print shapes and a few values** (`x.shape`, `x[:5]`, `x.dtype`) right before the failing line.
4. **Make it small:** reproduce the problem on one epoch or one subject.
5. **Plot it.** Most signal bugs are obvious in a plot.
6. **Ask** in the *Capstone Q&A* discussion with the error message, what you tried, and your track.

## Before you hand in: checklist

- [ ] A fresh clone + `pip install -r requirements-lock.txt` + your README commands reproduces your numbers
- [ ] `cfg["seed"]` is set and used by every random step
- [ ] Every result in `RESULTS.md` says which seed, code revision and settings produced it
- [ ] Validation is split by your track's split unit, and you report the spread across folds
- [ ] No data or `.venv` in the repository

**Next:** Module 7 — saving results and figures, caching slow steps, and (optional) experiment tracking.